In [ ]:
#| default_exp llm_tools


# LLM Tools

In [ ]:
#| export
import requests
import litellm
import json
from lisette import lite_mk_func

TOOLS = {}
TOOL_SCHEMAS = []


def get_tools():
    """Return available tool schemas."""
    return TOOL_SCHEMAS


def register_tool(func):
    """Register a function as a tool."""
    TOOLS[func.__name__] = func
    schema = lite_mk_func(func)
    # Remove old schema if exists
    TOOL_SCHEMAS[:] = [s for s in TOOL_SCHEMAS if s['function']['name'] != func.__name__]
    TOOL_SCHEMAS.append(schema)
    return func


@register_tool
def search_web(query: str, max_results: int = 10):
    """Search DuckDuckGo and return top results.
    
    Args:
        query: Search string.
        max_results: Maximum number of results to return.
        
    Returns:
        JSON string of results with title, url, snippet.
    """    
    from ddgs import DDGS
    
    results = DDGS().text(query, max_results=max_results)
    return str([{"title": r["title"], "url": r["href"], "snippet": r["body"]} 
            for r in results])


@register_tool
def read_url(url:str):
    """Retrieve webpage HTML content.
    
    Args:
        url: URL to fetch.
        
    Returns:
        Raw HTML string.
    """    
    import requests
    response = requests.get(url)
    html = response.text
    return html
